
# 練習問題: 自動ベクトル化のレポートを読む (C と Fortran の違い)

## 目標

コンパイラの最適化レポート (`-Minfo`) を読み, ループがSIMD化されたかどうかを自分で確認できるようになる.
とくに **C/C++ ではポインタの「エイリアスの可能性」が自動ベクトル化の妨げになる** のに対し, **Fortran は配列引数が重ならないと仮定するので何もしなくてもSIMD化されやすい** という違いを体験する.

これは穴埋め問題ではない. コンパイラの出力を読んで考える問題である.

## 課題

`autovec.cpp` (または `autovec.f90`) は saxpy (`y[i] = a*x[i] + y[i]`) を行う.

### 手順1: まず動かす

```
nvc++ -fast autovec.cpp -o autovec.exe   # C++
nvfortran -fast autovec.f90 -o autovec.exe  # Fortran
./autovec.exe
```

### 手順2: 最適化レポートを読む

`saxpy` 関数だけをオブジェクトコンパイル (`-c`) し, `-Minfo`/`-Mneginfo` のレポートと生成アセンブリ (`-Mkeepasm` で `.s`) を見る.

```
# C++
nvc++ -fast -Minfo -Mneginfo -Mkeepasm -c autovec.cpp
# Fortran
nvfortran -fast -Minfo -Mneginfo -Mkeepasm -c autovec.f90
```

- **C++** では, ループはベクトル化される一方で「エイリアスの可能性があるため版を分けた (versioned for ...)」旨の報告が出ることがある. これは, 実行時に `x` と `y` が重なっていないか確認してから速い版を使う, という安全策である.
- **Fortran** では, そうした版分けの注記なしに素直にベクトル化される (配列引数は重ならない前提のため).

`.s` の中に `pd` の付いた packed double 命令 (`vmulpd`, `vaddpd`, `vfmadd...pd`) が出ているかも確認せよ.

### 手順3 (工夫): C++ で版分けを無くす

C++ 版の `saxpy` のポインタに `__restrict` を付ける, または ループ直前に `#pragma omp simd` を加える (後者は `-mp=multicore` が必要) と, 「重ならない」とコンパイラに伝えられ, エイリアスのための版分けが消える. 前後で `-Minfo` の出力がどう変わるか比べよ.

```cpp
void saxpy(long n, double a, double * __restrict x, double * __restrict y) { ... }
```

## 期待される結果・考察

- C++ は (何も言わないと) エイリアスの可能性に備えるが, `__restrict` や `#pragma omp simd` で解消できる.
- Fortran は最初からその心配が無く, 自動ベクトル化される.
- 「同じ計算でも, 言語の前提 (エイリアスの扱い) によってコンパイラの仕事が変わる」ことを理解せよ.



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ autovec.cpp
#include <cstdio>
#include <cstdlib>

/* saxpy: y[i] = a*x[i] + y[i].
   x と y がポインタなので, コンパイラは「x と y が重なって (エイリアスして) いるかも
   しれない」と考えざるを得ない. そのため自動ベクトル化はできても, 実行時に重なりを
   確認して2通りのコードに分岐する (versioning) ことがある. */
void saxpy(long n, double a, double * x, double * y) {
  for (long i = 0; i < n; i++) {
    y[i] = a * x[i] + y[i];
  }
}

int main(int argc, char ** argv) {
  long n = (argc > 1 ? atol(argv[1]) : 16);
  double * x = (double *)malloc(sizeof(double) * n);
  double * y = (double *)malloc(sizeof(double) * n);
  for (long i = 0; i < n; i++) { x[i] = i; y[i] = 0.0; }
  saxpy(n, 2.0, x, y);
  printf("y[0] = %.1f, y[%ld] = %.1f\n", y[0], n - 1, y[n - 1]);
  free(x); free(y);
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore autovec.cpp -o autovec_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./autovec_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ autovec.f90
! saxpy: y(i) = a*x(i) + y(i).
! Fortran では配列引数 x, y は「重ならない (エイリアスしない)」と仮定してよいので,
! コンパイラは何も指示しなくても素直に自動ベクトル化できる (versioning も不要).
subroutine saxpy(n, a, x, y)
  integer(8), intent(in) :: n
  real(8), intent(in) :: a, x(n)
  real(8), intent(inout) :: y(n)
  integer(8) :: i
  do i = 1, n
     y(i) = a * x(i) + y(i)
  end do
end subroutine saxpy

program autovec
  character(len=32) :: arg
  integer(8) :: n, i
  real(8), allocatable :: x(:), y(:)
  n = 16
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg); read (arg, *) n
  end if
  allocate(x(n), y(n))
  do i = 1, n
     x(i) = i - 1; y(i) = 0.0d0
  end do
  call saxpy(n, 2.0d0, x, y)
  print "(a,f0.1,a,i0,a,f0.1)", "y(1) = ", y(1), ", y(", n, ") = ", y(n)
end program autovec

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore autovec.f90 -o autovec_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./autovec_f90.exe


# 4. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:autovec.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 4-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 4-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 4-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:autovec.cpp}

コマンドと実行結果:
{bash[-1]}

## 4-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:autovec.cpp}